# Ocular Disease Recognition — Grouped Classification

Same EfficientNetB0 transfer learning pipeline as `02_model_training.ipynb`, reformulated as a
**4-group clinical taxonomy** instead of 8 individual diagnoses.

The grouping absorbs the statistically unreliable Hypertension class (103 training samples) into
the Vascular group and reduces the imbalance ratio from 5.5:1 to ~1.2:1 after augmentation.

| Group | Classes | Clinical rationale |
|---|---|---|
| Normal | N | Healthy retina |
| Vascular / Metabolic | D + H | Vascular damage from elevated glucose or blood pressure |
| Structural / Mechanical | G + C + M | Physical changes: optic disc, lens opacity, axial elongation |
| Degenerative / Other | A + O | Macular degeneration and heterogeneous conditions |

# Environment Setup (Google Colab)

Uncomment and run the block below **only** when running on Google Colab. For local execution, skip this cell entirely.

In [ ]:
# ── Run this block ONLY when using Google Colab ────────────────────────
# 1. Upload train_data.h5 and test_data.h5 to Google Drive at:  My Drive/ocular/
# 2. For images choose Option A (Drive) or Option B (Kaggle API)
# 3. Uncomment the entire block and run before any other cell

# import os, shutil
# from google.colab import drive
# drive.mount('/content/drive')

# DRIVE_DIR = '/content/drive/MyDrive/ocular'

# os.makedirs('/content/Main', exist_ok=True)
# os.makedirs('/content/models', exist_ok=True)
# os.makedirs('/content/notebooks', exist_ok=True)

# shutil.copy(f'{DRIVE_DIR}/train_data.h5', '/content/Main/train_data.h5')
# shutil.copy(f'{DRIVE_DIR}/test_data.h5',  '/content/Main/test_data.h5')
# os.chdir('/content/notebooks')

# # Option A — images already uploaded to Drive:
# IMAGE_FOLDER_PATH = f'{DRIVE_DIR}/preprocessed_images'

# # Option B — download from Kaggle:
# shutil.copy(f'{DRIVE_DIR}/kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
# os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
# os.system('kaggle datasets download -d andrewmvd/ocular-disease-recognition-odir5k -p /content/preprocessed_images --unzip')
# IMAGE_FOLDER_PATH = '/content/preprocessed_images'

# import tensorflow as tf
# print('GPU:', tf.config.list_physical_devices('GPU'))


# 1. Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.metrics import accuracy_score, log_loss
from sklearn.utils import class_weight

import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


# 2. Data Loading

In [ ]:
train_data_path = '../Main/train_data.h5'
test_data_path  = '../Main/test_data.h5'

train_data = pd.read_hdf(train_data_path)
test_data  = pd.read_hdf(test_data_path)
train_data.head()


## 2.1 Feature / Target Split

In [ ]:
X_train, y_train = train_data.drop(columns=['target']), train_data['target']
X_test,  y_test  = test_data.drop(columns=['target']),  test_data['target']
print('8-class distribution (train):')
print(y_train.value_counts().sort_index())


## 2.2 Clinical Group Mapping

The 8 individual diagnoses are merged into 4 clinically coherent groups based on shared
aetiology and visual presentation in fundus photography.

| Group | Label | Original classes | Rationale |
|---|---|---|---|
| Normal | 0 | N | Healthy retina |
| Vascular / Metabolic | 1 | D + H | Microaneurismas, haemorrhages, exudates from elevated glucose or BP |
| Structural / Mechanical | 2 | G + C + M | Optic disc excavation, lens opacity, posterior pole stretching |
| Degenerative / Other | 3 | A + O | Drusen, macular changes, heterogeneous conditions |

In [ ]:
GROUP_MAP = {
    0: 0,  # Normal
    1: 1,  # Diabetic Retinopathy  -> Vascular/Metabolic
    5: 1,  # Hypertension          -> Vascular/Metabolic
    2: 2,  # Glaucoma              -> Structural/Mechanical
    3: 2,  # Cataract              -> Structural/Mechanical
    6: 2,  # Myopia                -> Structural/Mechanical
    4: 3,  # AMD                   -> Degenerative/Other
    7: 3,  # Other                 -> Degenerative/Other
}
GROUP_NAMES = ['Normal', 'Vascular/Metabolic', 'Structural/Mechanical', 'Degenerative/Other']

y_train_grouped = y_train.map(GROUP_MAP)
y_test_grouped  = y_test.map(GROUP_MAP)

print('Group distribution (train):')
for g, name in enumerate(GROUP_NAMES):
    n = (y_train_grouped == g).sum()
    print(f'  {g} {name}: {n}')
print(f'\nImbalance ratio: {y_train_grouped.value_counts().max() / y_train_grouped.value_counts().min():.2f}:1')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

orig_counts = y_train.value_counts().sort_index()
orig_labels = ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']
axes[0].bar(orig_labels, orig_counts, color='#4C9BE8')
axes[0].set_title('Original 8-class distribution (train)')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Samples')

group_counts = y_train_grouped.value_counts().sort_index()
axes[1].bar(GROUP_NAMES, group_counts, color=['#4C9BE8', '#E8884C', '#4CE8A0', '#E84C7A'])
axes[1].set_title('Clinical group distribution (train)')
axes[1].set_xlabel('Group')
axes[1].set_ylabel('Samples')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('../figures/group_distribution.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.3 Load Images

Images resized to **224×224** (EfficientNetB0 native resolution). Passed as `float32 [0–255]`
without manual normalisation — EfficientNetB0 includes a built-in rescaling layer.

In [ ]:
if 'IMAGE_FOLDER_PATH' not in vars():
    IMAGE_FOLDER_PATH = '../preprocessed_images'
IMAGE_SIZE = (224, 224)

def load_images(filenames):
    images = []
    for filename in filenames:
        img_path = os.path.join(IMAGE_FOLDER_PATH, filename)
        img = cv2.imread(img_path)
        if img is not None:
            img = cv2.resize(img, IMAGE_SIZE)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img.astype("float32")
            images.append(img)
        else:
            print(f"Warning: image {filename} could not be loaded.")
    return np.array(images)

X_train_images = load_images(X_train['filename'])
print(f'Loaded {X_train_images.shape[0]} training images at {IMAGE_SIZE}')


# 3. Model Architecture

**EfficientNetB0** (ImageNet pre-trained, frozen) with a **4-class** classification head:

```
GlobalAveragePooling2D → Dense(256, ReLU, L2=0.01) → Dropout(0.5) → Dense(4, softmax)
```

In [ ]:
def build_transfer_model(input_shape=(224, 224, 3), num_classes=4):
    base_model = tf.keras.applications.EfficientNetB0(
        input_shape=input_shape,
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation="relu",
                               kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs), base_model

model, base_model = build_transfer_model()
model.summary()


In [ ]:
y_train_grouped = y_train_grouped.reset_index(drop=True)
print(y_train_grouped.head())


# 4. Validation Split

Stratified 20% per group extracted from **original** images **before** augmentation.
`np.random.seed(511)` ensures reproducibility across runs.

In [ ]:
np.random.seed(511)
validation_indices = []
for grp in sorted(y_train_grouped.unique()):
    grp_indices = y_train_grouped[y_train_grouped == grp].index
    n_val = max(1, int(len(grp_indices) * 0.2))
    chosen = np.random.choice(grp_indices, n_val, replace=False)
    validation_indices.extend(chosen)

validation_indices = np.array(validation_indices)

X_val          = X_train_images[validation_indices]
y_val          = y_train_grouped[validation_indices]

X_train_images = np.delete(X_train_images, validation_indices, axis=0)
y_train_grouped = y_train_grouped.drop(validation_indices).reset_index(drop=True)

print(f"Validation set: {X_val.shape[0]}  |  Training set: {X_train_images.shape[0]}")
print("Groups in validation:", dict(pd.Series(y_val.values).value_counts().sort_index()))


# 5. Data Augmentation

Tiered strategy targeting ~1.2:1 imbalance ratio after augmentation.

| Group | Original | Copies | Total |
|---|---|---|---|
| Normal (0) | ~1839 | 0 | ~1839 |
| Vascular (1) | ~1111 | 1 | ~2222 |
| Structural (2) | ~518 | 3 | ~2072 |
| Degenerative (3) | ~623 | 2 | ~1869 |

**Resulting imbalance ratio: ~1.2:1** (down from 5.5:1).

In [ ]:
def augment_images(images, labels):
    def make_transforms(img):
        return [
            tf.image.flip_left_right(img),
            tf.image.flip_up_down(img),
            tf.image.flip_left_right(tf.image.flip_up_down(img)),
            tf.image.rot90(img, k=1),
            tf.image.rot90(img, k=2),
            tf.image.rot90(img, k=3),
            tf.image.random_brightness(img, max_delta=0.2),
            tf.image.random_contrast(img, lower=0.8, upper=1.2),
            tf.image.flip_left_right(tf.image.rot90(img, k=1)),
            tf.image.flip_left_right(tf.image.rot90(img, k=3)),
        ]

    copies_per_class = {0: 0, 1: 1, 2: 3, 3: 2}

    augmented_images, augmented_labels = [], []
    for img, label in zip(images, labels):
        n_copies = copies_per_class[label]
        if n_copies > 0:
            t = make_transforms(img)
            for i in range(n_copies):
                augmented_images.append(t[i])
                augmented_labels.append(label)
    return np.array(augmented_images), np.array(augmented_labels)


In [ ]:
X_train_augmented, y_train_augmented = augment_images(X_train_images, y_train_grouped)
print(f"Augmented: {X_train_augmented.shape[0]}  |  Training originals: {X_train_images.shape[0]}")


## 5.1 Augmented Sample Images

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for i in range(5):
    idx = np.random.randint(0, len(X_train_augmented))
    axes[i].imshow(X_train_augmented[idx].astype(np.uint8))
    axes[i].set_title(f"Group {y_train_augmented[idx]}: {GROUP_NAMES[y_train_augmented[idx]]}")
    axes[i].axis("off")
plt.suptitle("Augmented training samples")
plt.show()


## 5.2 Combined Class Distribution

In [ ]:
print("Original training set:")
print(y_train_grouped.value_counts().sort_index())

unique, counts = np.unique(y_train_augmented, return_counts=True)
print("\nAugmented additions:")
for u, c in zip(unique, counts):
    print(f"  Group {u} ({GROUP_NAMES[u]}): +{c}")


## 5.3 Combine Original and Augmented

In [ ]:
X_train_combined = np.concatenate((X_train_images, X_train_augmented), axis=0)
y_train_combined = np.concatenate((y_train_grouped, y_train_augmented), axis=0)
print(f"Total training samples: {X_train_combined.shape[0]}")
print("Final distribution:")
for g, name in enumerate(GROUP_NAMES):
    n = (y_train_combined == g).sum()
    print(f"  {g} {name}: {n}")
min_count = min((y_train_combined == g).sum() for g in range(4))
max_count = max((y_train_combined == g).sum() for g in range(4))
print(f"Imbalance ratio: {max_count / min_count:.2f}:1")


# 6. Training — Phase 4 (Frozen Base)

Train only the classification head. Balanced class weights compensate for residual imbalance.
`ReduceLROnPlateau` halves lr after 5 stagnant epochs; `EarlyStopping` (patience=25)
restores the best checkpoint.

In [ ]:
class_weights_array = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_combined),
    y=y_train_combined
)
class_weights_dict = dict(enumerate(class_weights_array))
print("Class weights:", {k: round(v, 3) for k, v in class_weights_dict.items()})

early_stopping = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_combined, y_train_combined,
    epochs=120,
    validation_data=(X_val, y_val),
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weights_dict
)


In [ ]:
model.save('../models/efficientnetb0_grouped_phase4.h5')
print('Phase 4 model saved -> models/efficientnetb0_grouped_phase4.h5')


# 7. Evaluation — Phase 4

In [ ]:
X_test_images = load_images(X_test['filename'])
print(f'Loaded {X_test_images.shape[0]} test images')


## 7.1 Confusion Matrix

In [ ]:
y_pred = model.predict(X_test_images)
y_pred_classes = np.argmax(y_pred, axis=1)

cm = confusion_matrix(y_test_grouped, y_pred_classes)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=GROUP_NAMES)
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix — Phase 4 (Grouped)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 7.2 Loss & Accuracy

In [ ]:
accuracy = accuracy_score(y_test_grouped, y_pred_classes)
loss = log_loss(y_test_grouped, y_pred)
print(f"Loss:     {loss:.4f}")
print(f"Accuracy: {accuracy:.4f}")


## 7.3 Per-Class Report

In [ ]:
print(classification_report(y_test_grouped, y_pred_classes, target_names=GROUP_NAMES))


# 8. Training — Phase 5 (Fine-tuning)

Unfreeze the **last 20 layers** of EfficientNetB0 and fine-tune at lr=1e-5 to adapt
pre-trained features to the retinal domain.

> **Requires GPU.** In Google Colab: *Runtime → Change runtime type → T4 GPU*.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

print(f"Trainable layers: {sum(1 for l in base_model.layers if l.trainable)}/{len(base_model.layers)}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

fine_tuning_stopping = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
fine_tuning_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, min_lr=1e-8, verbose=1)

history_ft = model.fit(
    X_train_combined, y_train_combined,
    epochs=100,
    validation_data=(X_val, y_val),
    batch_size=16,
    callbacks=[fine_tuning_stopping, fine_tuning_lr],
    class_weight=class_weights_dict
)


## 8.1 Evaluation — Phase 5

In [ ]:
y_pred_ft = model.predict(X_test_images)
y_pred_ft_classes = np.argmax(y_pred_ft, axis=1)

cm_ft = confusion_matrix(y_test_grouped, y_pred_ft_classes)
disp_ft = ConfusionMatrixDisplay(confusion_matrix=cm_ft, display_labels=GROUP_NAMES)
disp_ft.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix — Phase 5 Fine-tuning (Grouped)')
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

accuracy_ft = accuracy_score(y_test_grouped, y_pred_ft_classes)
loss_ft = log_loss(y_test_grouped, y_pred_ft)
print(f"Loss:     {loss_ft:.4f}")
print(f"Accuracy: {accuracy_ft:.4f}")

print()
print(classification_report(y_test_grouped, y_pred_ft_classes, target_names=GROUP_NAMES))


In [ ]:
model.save('../models/efficientnetb0_grouped_finetuned.keras')
print('Fine-tuned model saved -> models/efficientnetb0_grouped_finetuned.keras')
